[Reference](https://medium.com/@GaoDalie_AI/dspark-rag-deepseek-just-made-every-llm-way-faster-open-source-1e29350a8ae3$0)

# DSpark

In [1]:
from concurrent.futures import ThreadPoolExecutor
from typing import Any, List, Optional

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from pydantic import PrivateAttr

from mlx_dspark import load_dflash_pair, dflash_generate


class ChatDFlash(BaseChatModel):
    """
    LangChain chat model backed by DFlash speculative decoding (mlx-dspark).

    All MLX work (model load + generation) is pinned to a single worker
    thread because MLX GPU streams are thread-bound and Streamlit executes
    each rerun in a fresh thread.
    """

    model_id: str = "mlx-community/gemma-4-12B-it-8bit"
    max_tokens: int = 1024
    temperature: float = 0.0

    _executor: Any = PrivateAttr()
    _target: Any = PrivateAttr()
    _tok: Any = PrivateAttr()
    _drafter: Any = PrivateAttr()
    _cfg: Any = PrivateAttr()

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._executor = ThreadPoolExecutor(max_workers=1)
        # Load inside the worker thread so the GPU stream lives there
        self._target, self._tok, self._drafter, self._cfg = self._executor.submit(
            load_dflash_pair, self.model_id
        ).result()

    @property
    def _llm_type(self) -> str:
        return "dflash-mlx"

    @staticmethod
    def _to_chat_dicts(messages: List[BaseMessage]) -> List[dict]:
        # Gemma's chat template rejects the system role -> fold into first user msg
        role_map = {"human": "user", "ai": "assistant", "system": "system"}
        out, system_buf = [], []
        for m in messages:
            role = role_map.get(m.type, "user")
            content = m.content if isinstance(m.content, str) else str(m.content)
            if role == "system":
                system_buf.append(content)
            else:
                out.append({"role": role, "content": content})
        if system_buf:
            if out:
                out[0]["content"] = "\n".join(system_buf) + "\n\n" + out[0]["content"]
            else:
                out.append({"role": "user", "content": "\n".join(system_buf)})
        return out

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager=None,
        **kwargs,
    ) -> ChatResult:
        prompt = self._tok.apply_chat_template(
            self._to_chat_dicts(messages),
            tokenize=False,
            add_generation_prompt=True,
        )
        res = self._executor.submit(
            dflash_generate,
            self._target,
            self._tok,
            self._drafter,
            prompt,
            max_new_tokens=self.max_tokens,
            temperature=self.temperature,
        ).result()
        msg = AIMessage(
            content=res.text,
            response_metadata={
                "mean_accept_len": res.mean_accept_len,
                "tokens_per_sec": res.tokens_per_sec,
            },
        )
        return ChatResult(generations=[ChatGeneration(message=msg)])

# RAG


In [2]:
from pathlib import Path
from typing import List

from langchain.schema import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain.schema import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from  dflash_llm import ChatDFlash
from vector_store import VectorStore


class LLMRAGHandler:
    """
    A class to handle LLM-based RAG (Retrieval-Augmented Generation) tasks.

    Attributes:
        llm (ChatDFlash): DFlash-accelerated language model (MLX, Apple Silicon).
        vector_store (VectorStore): The vector store used for document retrieval.
        system_prompt (str): The system prompt given to the model.
        history (List[BaseMessage]): The conversation history.
        rag_prompt (PromptTemplate): The prompt template for Q&A with RAG.
        llm_chain (Chain): prompt | llm | parser chain.

    Methods:
        generate_response(human_message) -> str: Generates and appends a response.
        reset() -> None: Resets the conversation history.
        get_history() -> List[BaseMessage]: Returns the conversation history.
        retrieve(question, k=4) -> List[Document]: Retrieves relevant documents.
        add_pdf_to_context(filePath): Adds a PDF file to the retrieval context.
    """

    def __init__(self, model="mlx-community/gemma-4-12B-it-8bit"):
        """
        Initializes the LLMRAGHandler with the specified model.

        Args:
            model (str): HuggingFace model id of the DFlash target model.
        """
        self.llm = ChatDFlash(model_id=model)
        self.vector_store = VectorStore()

        # System prompt - These are the instructions for the model
        self.system_prompt = (
            "You are an assistant for question-answering tasks."
            " Use the following pieces of retrieved context to answer the question."
            " If you don't know the answer, try to answer the question without"
            " context but mention that the context does not provide enough"
            " information. Use three sentences maximum and keep the answer concise."
        )

        # keep track of the conversation history
        self.history = []
        self.history.append(SystemMessage(content=self.system_prompt))

        # prompt template for q&a with rag
        self.rag_prompt = PromptTemplate.from_template(
            "Previous conversation: {chat_history}"
            " Question: {input}"
            " Context: {context}"
            " Answer:"
        )

        # Chain for querying the LLM and getting the answer
        self.llm_chain = self.rag_prompt | self.llm | StrOutputParser()

    def generate_response(self, human_message) -> str:
        """
        Generates and appends a response from the LLM.

        Args:
            human_message (str): The user's message.

        Returns:
            str: The AI's response text.
        """
        print("Adding Human Message...")
        print(f"{human_message}")

        print("Generating response from LLM...")
        context_docs = self.retrieve(human_message)
        context = "\n\n".join(d.page_content for d in context_docs)

        response = self.llm_chain.invoke(
            {
                "input": human_message,
                "context": context,
                "chat_history": self.history,
            }
        )
        print(response)

        self.history.append(HumanMessage(content=human_message))
        self.history.append(AIMessage(content=response))
        return response

    def reset(self) -> None:
        """
        Resets the conversation history.
        """
        self.history = []
        self.history.append(SystemMessage(content=self.system_prompt))

    def get_history(self) -> List[BaseMessage]:
        """
        Returns the conversation history.

        Returns:
            List[BaseMessage]: The conversation history.
        """
        return self.history

    def retrieve(self, question: str, k: int = 4) -> List[Document]:
        """
        Retrieves the most relevant documents for a given question.

        Args:
            question (str): The question to retrieve documents for.
            k (int): The number of documents to retrieve. Default is 4.

        Returns:
            List[Document]: The retrieved documents.
        """
        retrieved_docs = self.vector_store.similarity_search(question, k=k)
        return retrieved_docs

    def add_pdf_to_context(self, filePath: Path) -> List[Document]:
        """
        Adds a PDF file to the context for retrieval.

        Args:
            filePath (Path): The path to the PDF file.

        Returns:
            List[Document]: The documents added to the vector store.
        """
        return self.vector_store.add_document(filePath)